In [137]:
#Import/reload modules
import SparkDataCheck
from pyspark.sql import SparkSession
import pandas as pd
import importlib
importlib.reload(SparkDataCheck)

<module 'SparkDataCheck' from '/home/jupyter-mscampb4@ncsu.edu/Project2/SparkDataCheck.py'>

In [101]:
#Instantiate SparkDataCheck using air.csv
spark = SparkSession.builder.master('local[*]').appName('Project2').getOrCreate() #Initialize pyspark session
air = SparkDataCheck.SparkDataCheck.load_pyspark(spark, "air.csv")
air.df.show()

+---+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|          948|    172|        1092|    122|        1584| 

26/03/21 09:52:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-mscampb4@ncsu.edu/Project2/air.csv


In [135]:
#Alternatively, load in using a pandas df and initialize the class in this manner
airPd = pd.read_csv("air.csv")

#Do some data cleaning while we are here
airPd = airPd.replace([-200], [None])
airPd['Date'] = airPd.Date.astype('string')
airPd = airPd.rename(columns={'PT08.S1(CO)': 'PT08S1(CO)', 
                      'PT08.S2(NMHC)': 'PT08S2(NMHC)', 
                      'PT08.S3(NOx)': 'PT08S3(NOx)',
                      'PT08.S4(NO2)': 'PT08S4(NO2)',
                      'PT08.S5(O3)': 'PT08S5(O3)'}) #Removing the dots as they lead to errors

#Load it into pyspark
air = SparkDataCheck.SparkDataCheck.load_pandas(spark, airPd)

In [138]:
#Validate various columns for numeric values
air.validate_numeric("Date", lower = 0, upper = 100).show() #Returns error due to incorrect col type
air.validate_numeric("CO(GT)", lower = 0, upper = 100).show() #Evaluates
air.validate_numeric("CO(GT)", lower = 0).show() #Evaluates
air.validate_numeric("CO(GT)", upper = 100).show() #Evaluates
air.validate_numeric("CO(GT)").show() #Returns error due to lack of bounds

Please provide a numeric column.
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      1376|      80|     9.2|      

In [5]:
#Validate various columns for string values
validLevels = ["3/10/2004"]
air.validate_string("CO(GT)", validLevels).show() #Error due to ineligible data type
air.validate_string("Date", validLevels).show() #evaluates as expected

Please provide a string column.
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      

In [6]:
#Test denote_null_values method
air.denote_null_values("CO(GT)").show() #evaluates as expected

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|has_null_value|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         false|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|         false|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|  

In [123]:
#Test get_min_max
air.get_min_max(col = "CO(GT)") #Returns a dataframe with one row
air.get_min_max(col = "CO(GT)", groupCol = "Date").head() #Returns a dataframe with one row per date
air.get_min_max() #Returns a minimum and maximum for every numeric column
air.get_min_max(groupCol = "Date").head() #Returns a minimum and maximum for every numeric column with one row per date
air.get_min_max(col = "Date") #Returns an error due to incorrect data type

Please provide a numeric column.


DataFrame[Unnamed: 0: bigint, Date: string, Time: string, CO(GT): double, PT08S1(CO): bigint, NMHC(GT): bigint, C6H6(GT): double, PT08S2(NMHC): bigint, NOx(GT): bigint, PT08S3(NOx): bigint, NO2(GT): bigint, PT08S4(NO2): bigint, PT08S5(O3): bigint, T: double, RH: double, AH: double]

In [140]:
#Test get_count
air.get_count(col = "CO(GT)") #Returns error due to non-string data type
air.get_count(col = "Date").head() #Gets number of observations per date
air.get_count(col = "Date", groupCol = "CO(GT)") #Returns an error due to non-string data type
air.get_count(col = "Date", groupCol = "Time").head() #Gets number of observations per date/time combo

Please provide a string column.
Please provide a string column for the grouping variable.


,Date,Time,count(Date)
0,3/12/2004,11:00:00,1
1,3/11/2004,21:00:00,1
2,3/13/2004,0:00:00,1
3,3/12/2004,7:00:00,1
4,3/11/2004,18:00:00,1
